# Study 835 — Spurious Regression — the teardown

The oversized level-OLS *t* and R², the differencing fix, the trending case, the √T sample-size divergence, the stationary-series size control, the Engle-Granger cointegration positive control, and the costed pairs timer. Frozen headline in `R`; the live cells re-run the fast controls.

In [1]:
R = {'fingerprint': '73e2821b184c', 'base_seed': 835, 'n_pairs': 5000, 'n_obs': 250, 'lvl_reject': 0.85, 'lvl_reject_x': 17.0, 'lvl_wilson': (0.84, 0.86), 'lvl_mean_abs_t': 8.99, 'lvl_median_abs_t': 7.06, 'lvl_mean_r2': 0.241, 'lvl_share_r2': 0.398, 'dif_reject': 0.053, 'dif_mean_abs_t': 0.8, 'dif_mean_r2': 0.004, 'drift_reject': 0.981, 'drift_mean_abs_t': 28.48, 'drift_mean_r2': 0.662, 'drift_share_r2': 0.899, 'sweep': [(50, 0.679, 3.99, 0.243, 0.059), (125, 0.787, 6.2, 0.236, 0.052), (250, 0.847, 8.99, 0.241, 0.052), (500, 0.895, 12.81, 0.241, 0.05), (1000, 0.926, 17.99, 0.24, 0.045)], 'stat_reject': 0.051, 'stat_mean_abs_t': 0.8, 'stat_mean_r2': 0.004, 'coint_indep_reject': 0.05, 'coint_indep_p': 0.495, 'coint_true_reject': 1.0, 'coint_true_p': 0.0, 'timer': [(0.0, -27.46, -27.46, -1.23, -1.43, -69.2), (1.0, -27.46, -27.99, -1.25, -1.45, -70.5), (5.0, -27.46, -29.59, -1.33, -1.54, -74.6)]}

## 1. The pitfall vs the fix — level OLS on two independent random walks

5,000 pairs × 250 obs, driftless. Nominal test size is 0.05.

In [2]:
print(f"levels      : reject {R['lvl_reject']:.3f} (Wilson 95% CI {R['lvl_wilson']}), "
      f"{R['lvl_reject_x']:.1f}x oversized")
print(f"              mean|t| {R['lvl_mean_abs_t']:.2f}  median|t| {R['lvl_median_abs_t']:.2f}  "
      f"meanR2 {R['lvl_mean_r2']:.3f}  shareR2>0.25 {R['lvl_share_r2']:.3f}")
print(f"differences : reject {R['dif_reject']:.3f} (~nominal), mean|t| {R['dif_mean_abs_t']:.2f}, "
      f"meanR2 {R['dif_mean_r2']:.3f}  <- the fix")

levels      : reject 0.850 (Wilson 95% CI (0.84, 0.86)), 17.0x oversized
              mean|t| 8.99  median|t| 7.06  meanR2 0.241  shareR2>0.25 0.398
differences : reject 0.053 (~nominal), mean|t| 0.80, meanR2 0.004  <- the fix


## 2. Trending series manufacture false significance (drift = 0.15/step)

In [3]:
print(f"trending levels: reject {R['drift_reject']:.3f}, mean|t| {R['drift_mean_abs_t']:.2f}, "
      f"meanR2 {R['drift_mean_r2']:.3f}, shareR2>0.25 {R['drift_share_r2']:.3f}")

trending levels: reject 0.981, mean|t| 28.48, meanR2 0.662, shareR2>0.25 0.899


## 3. The √T divergence — more data makes the LEVEL test worse

Phillips (1986): with `I(1)` regressors the *t*-stat diverges, so the rejection rate → 1 as `n` grows. The differenced test stays correctly sized.

In [4]:
print('n_obs | level_reject  level_mean|t|  level_R2 | diff_reject')
for n, lr, mt, r2, dr in R['sweep']:
    print(f'{n:>5} | {lr:.3f}        {mt:>5.2f}        {r2:.3f} | {dr:.3f}')

n_obs | level_reject  level_mean|t|  level_R2 | diff_reject
   50 | 0.679         3.99        0.243 | 0.059
  125 | 0.787         6.20        0.236 | 0.052
  250 | 0.847         8.99        0.241 | 0.052
  500 | 0.895        12.81        0.241 | 0.050
 1000 | 0.926        17.99        0.240 | 0.045


## 4. Specificity control — the same OLS on STATIONARY series is correctly sized

Live: the over-rejection is a property of the unit root, not of OLS.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from spurious_regression import data, strategy as st
sc = st.size_control(data, n_pairs=3000, n_obs=250, phi=0.0, seed=835)
print(f"stationary levels: reject {sc['reject_rate']:.3f} (~0.05, correctly sized), "
      f"mean|t| {sc['mean_abs_t']:.2f}, meanR2 {sc['mean_r2']:.3f}")
print(f"(frozen headline at 5,000 pairs: {R['stat_reject']:.3f})")

stationary levels: reject 0.047 (~0.05, correctly sized), mean|t| 0.80, meanR2 0.004
(frozen headline at 5,000 pairs: 0.051)


## 5. The other fix — Engle-Granger cointegration (positive control)

Live, on a small sample: the test must NOT reject on independent walks and MUST reject on a genuinely cointegrated pair.

In [6]:
Xi, Yi = data.independent_walks(120, n_obs=250, seed=835)
Xc, Yc = data.cointegrated_pairs(120, n_obs=250, beta=1.0, noise_sd=1.0, seed=835)
ci = st.cointegration_reject_rate(Xi, Yi)
cc = st.cointegration_reject_rate(Xc, Yc)
print(f"independent walks : reject no-coint {ci['reject_rate']:.3f} (median p {ci['median_pvalue']:.3f}) -> nothing")
print(f"cointegrated pair : reject no-coint {cc['reject_rate']:.3f} (median p {cc['median_pvalue']:.3f}) -> real relation")
print(f"(frozen headline, 300 pairs: indep {R['coint_indep_reject']:.3f} / true {R['coint_true_reject']:.3f})")

independent walks : reject no-coint 0.050 (median p 0.398) -> nothing
cointegrated pair : reject no-coint 1.000 (median p 0.000) -> real relation
(frozen headline, 300 pairs: indep 0.050 / true 1.000)


## 6. Tradability — a costed pairs trade on the spurious spread (no look-ahead)

Trailing hedge ratio & z-score known at `t−1`; contrarian on the residual; one-way cost × NAV on turnover + short borrow.

In [7]:
print('cost  | gross  ->  net  | t_net | Sharpe |  ~ann')
for c, g, n, t, sh, ann in R['timer']:
    print(f'{c:>4.1f} | {g:+.2f} -> {n:+.2f} | {t:+.2f} | {sh:+.2f} | {ann:+.1f}%')
print('\n-> gross |t|<2: the spread is a random walk, no reversion to harvest; costs only hurt. MIRAGE.')

cost  | gross  ->  net  | t_net | Sharpe |  ~ann
 0.0 | -27.46 -> -27.46 | -1.23 | -1.43 | -69.2%
 1.0 | -27.46 -> -27.99 | -1.25 | -1.45 | -70.5%
 5.0 | -27.46 -> -29.59 | -1.33 | -1.54 | -74.6%

-> gross |t|<2: the spread is a random walk, no reversion to harvest; costs only hurt. MIRAGE.


## Verdict

- **Signal — None.** The two series are drawn independent; the level regression's significance is a manufactured artefact of the unit root (reject **0.850** vs nominal 0.05, a **17×** oversized test; mean R² **0.24**). First-differencing restores the correct size (**0.053**) and a stationary-series control is correctly sized (**0.051**), so the inflation is nonstationarity, not OLS. No real tape a method demo could stamp → capped at None.
- **Tradability — Mirage.** The spurious spread is a random walk; the costed pairs trade earns no edge distinguishable from zero (gross *t* = **-1.23**) and loses net of any friction.
- **Do trending series manufacture false significance? — Confirmed.** 85% false rejection on driftless walks, **98%** with a shared trend, and the inflation *grows with the sample* (mean |t| 4.0 → 18.0 as n: 50 → 1000). The cointegration test tells the spurious from the genuine (reject 0.05 vs 1.00).